# **Filter-Based Methods:**

- Correlation-Based Analysis
- Mutual Information
- Chi-Squared Test
- Variance Threshold

In Feature Selection we're going to find a subset of the original features that make our ml task more efficient and accurate. Filter-based methods are a kind of approach in feature selection. They are the first baseline that we start using them when we wanna do a feature selection task.

**Pros:**
- Extendable for large datasets cause they are not computationaly expensive.
- Great baseline approach to reduce the dimensionality for more advanced methods that are computationaly expensive.

**Cons:**
- Don't consider interactions between features.
- They are more a univariate analysis.

## **Correlation-Based Analysis:**

One of the commong correlation-based filtering techniques is `Pearson Correlation`:

> **Note:** Pearson correlation in useful for dataset with numerical target variable.

> **Note:** To deal with datasets that have categorical target variable we can use Mutual Information (MI).

> **Note:** To deal with dataset with mixture of categorical and numerical features and a categorical target variable it's better to use Chi-Squared Test.

In [62]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.feature_selection import mutual_info_classif

In [68]:
diabetes_data = load_diabetes()
diabetes = pd.concat(
    [
        pd.DataFrame(wine_data['data'], columns=wine_data['feature_names']),
        pd.Series(wine_data['target'], name="target")
	],
    axis=1
)
diabetes

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0
...,...,...,...,...,...,...,...,...,...,...,...
437,0.041708,0.050680,0.019662,0.059744,-0.005697,-0.002566,-0.028674,-0.002592,0.031193,0.007207,178.0
438,-0.005515,0.050680,-0.015906,-0.067642,0.049341,0.079165,-0.028674,0.034309,-0.018114,0.044485,104.0
439,0.041708,0.050680,-0.015906,0.017293,-0.037344,-0.013840,-0.024993,-0.011080,-0.046883,0.015491,132.0
440,-0.045472,-0.044642,0.039062,0.001215,0.016318,0.015283,-0.028674,0.026560,0.044529,-0.025930,220.0


In [70]:
method = "pearson"

pearson_corr = abs(diabetes.drop("target", axis=1).corrwith(diabetes['target'], method=method)).sort_values(ascending=False)
pd.DataFrame(pearson_corr).rename({0: method + "_correlation"}, axis=1)

,pearson_correlation
bmi,0.586450
s5,0.565883
bp,0.441482
s4,0.430453
s3,0.394789
s6,0.382483
s1,0.212022
age,0.187889
s2,0.174054
sex,0.043062


In [71]:
def corr_analysis(
        df: pd.DataFrame,
        target_name: str,
        method: str = "pearson",
        threshold: float = 0
) -> pd.DataFrame:
    """Create a data frame of the Correlation analysis

    This function creates a correlation analysis data frame based_on the
    method you choose for the analysis and using threshold argument you can
    filter the results based-on the correlation value. 

    :param df: The input dataframe
    :param target_name: The target variable name
    :param method: The correlation analysis method name, defaults to "pearson"
    :param threshold: Threshold to filter the results, defaults to 0
    :return: The correlation analysis data frame
    """
    try:
        corr = abs(df.drop(target_name, axis=1).corrwith(
            df[target_name], method=method)).sort_values(ascending=False)
        corr_df = pd.DataFrame(corr).rename({0: f"{method}_correlation"}, axis=1)
        
        return corr_df.loc[corr_df[f"{method}_correlation"] > threshold]

    except ValueError:
        print("ValueError: Method argument not exist.")
        print("Methods to choose: ['pearson', 'kendall', 'spearman']")
        return pd.DataFrame()

In [73]:
corr_analysis_selected_featuers = corr_analysis(diabetes, target_name="target", method="pearson")
corr_analysis_selected_featuers

,pearson_correlation
bmi,0.586450
s5,0.565883
bp,0.441482
s4,0.430453
s3,0.394789
s6,0.382483
s1,0.212022
age,0.187889
s2,0.174054
sex,0.043062


The problem with this method is thet pearson correlation analysis can only handle continuous target variable and it doesn't consider the feature interactions and is a univariate analysis.

## **Mutual Information:**

Mutual information is a statistic property that indicates a how much does a feature have impact on the target variable predictability. High mutual information value indicates that the amount of information obtained from the target variable by observing the feature variable is high.

In [92]:
from sklearn.feature_selection import mutual_info_regression

X, y = diabetes_data.data, diabetes_data.target
mi_scores = mutual_info_classif(X, y)

mi_df = pd.DataFrame(
    data=mi_scores, index=diabetes_data.feature_names, columns=["mi-score"]
).sort_values(by="mi-score", axis=0, ascending=False)
mi_df

,mi-score
sex,1.052012
s4,0.336896
s5,0.188015
s3,0.158665
bmi,0.149761
bp,0.059062
age,0.057747
s1,0.000000
s2,0.000000
s6,0.000000


In [94]:
from sklearn.feature_selection import mutual_info_regression

X, y = diabetes_data.data, diabetes_data.target
mi_scores = mutual_info_regression(X, y)

mi_df = pd.DataFrame(
    data=mi_scores, index=diabetes_data.feature_names, columns=["mi-score"]
).sort_values(by="mi-score", axis=0, ascending=False)
mi_df

,mi-score
bmi,0.174317
s5,0.150138
s6,0.107184
s4,0.093011
s1,0.069137
s3,0.061252
bp,0.054068
sex,0.030459
s2,0.014082
age,0.013288


read scikit-learn documentation for: mutual_info_regression, mutual_info_classif